<a href="https://colab.research.google.com/github/pedrohgdp/BuildingALLM/blob/main/DatasAndEmbedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import tiktoken
from google.colab import files
from torch.utils.data import DataLoader, Dataset

# The dataset class save the data in the way that we can
# use.
# In this case we will save in tensor arrays.

class DataSet(Dataset):
  def __init__(self, text, tokenizer, context_size, stride):
    self.input_ids = []
    self.output_ids = []

    tokens_ids = tokenizer.encode(text)

    for i in range(0, len(tokens_ids) - context_size, stride):
      self.input_ids.append(torch.tensor(tokens_ids[i:i+context_size]))
      self.output_ids.append(torch.tensor(tokens_ids[i + 1:i + context_size + 1]))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
      return self.input_ids[idx], self.output_ids[idx]



# That function create the dataloader to take data from
# Dataset and this return of dataloader can be print or
# In the future, use as a entry of a neural network.

def create_data_loader(text, batch_size = 4,
                       context_size = 256,
                       stride = 128,
                       shuffle = True,
                       drop_last = True,
                       num_workers = 0):

  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = DataSet(text, tokenizer, context_size, stride)
  dataloader = DataLoader(
      dataset,
      batch_size = batch_size,
      shuffle = shuffle,
      drop_last = drop_last,
      num_workers = num_workers)

  return dataloader



# Opening the file and using dataset and dataloader.
text_file = files.upload()
filename = list(text_file.keys())[0]

# 3. Read the file
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# Batch is the size of the return
# Batch = 1 is 1 line
# Batch = 2 is 2 line
# go and go
dataloader = create_data_loader(text, batch_size=2, context_size=4, stride=1, shuffle=False)
data_iterator = iter(dataloader)

input, target = next(data_iterator)
print("The input is: ", input, "\n The target is: ", target)


Saving The_Verdict.txt to The_Verdict (1).txt
The input is:  tensor([[ 171,  119,  123,  464],
        [ 119,  123,  464, 4643]]) 
 The target is:  tensor([[  119,   123,   464,  4643],
        [  123,   464,  4643, 11600]])


# Dataset

At this point we already know how to create a vocabulary, split a text into tokens, and transform them into token_ids.

But now, instead of using the class we created before to generate token_ids, we will use Tiktoken library, which already has a large vocabulary and can tokenize the text for us (transforming tokens into token_ids).

Tiktoken deals with unknown words in a different way than the tokenizer I made.

In my tokenizer, we add a special token for unknown words. Tiktoken instead does not need this kind of token for most cases.

Because Tiktoken creates tokens by combining smaller pieces of words, it can represent new words using tokens that already exist in the vocabulary.

Tiktoken uses the BPE algorithm.

The `Dataset` is responsible for storing the data in a format that the `DataLoader`, which we will talk about later, can load.

In this case, the `Dataset` saves our token_ids as tensors.  
Tensors are multidimensional arrays that PyTorch can use to perform mathematical operations efficiently.

Inside the `Dataset` we will have two types of tensors:

- **Input**: Represents the input sequence we will give to the model.
- **Output**: Represents the expected output sequence for that input.

The input and output always have the same size, but the output sequence is shifted one token ahead of the input.

For example:

If `context_size = 1`:

```text
Input  = [10]
Output = [11]
```

If `context_size = 2`:

```text
Input  = [10, 11]
Output = [11, 12]
```

So the input uses the tokens from `i` to `i + context_size`, while the output starts at `i + 1` and goes until `i + context_size + 1`.

This allows the model to learn how to predict the next token in a sequence.

---

# DataLoader

The `DataLoader` takes the data stored inside the `Dataset` and returns it in batches.

A batch is a group of samples.

So when we set:

```python
batch_size = 4
```

the `DataLoader` will return 4 samples at once.

In our case, each sample contains:
- one input tensor
- one output tensor

Each tensor has the size defined by `context_size`.

# Embedding the tokens_ids

In [ ]:
vocab_size = 50257
dim_size = 4
context_length = 4

inputs, targets = next(data_iterator)

print("Tokens_ids:: ", inputs)
print("\n")
print("Size of inputs: ", inputs.shape)
print("\n")
#The inputs.shape will return (8, 4) because the batch size is 8 and
#the sequence length is 4
#Só every tensor have 4 tokens ids and we returning 8 of them

# Create embedding layers
token_embedding_layer = torch.nn.Embedding(vocab_size, dim_size)
print("Embedding_layer: \n", token_embedding_layer)
print("\n")
position_embedding_layer = torch.nn.Embedding(context_length, dim_size)
# The positional embedding layer will be a matrix with 4 rows
# because context_length is 4.

# Each row corresponds to a position in the sequence:
# position 0, position 1, position 2, and position 3.

# Each positional embedding has the same dimension as the token embeddings
# so both vectors can be summed.

# The same input that the print above
token_embeddings = token_embedding_layer(inputs)

# position id of each token
position_ids = torch.arange(inputs.shape[1])

position_embeddings = position_embedding_layer(position_ids)

final_embeddings = token_embeddings + position_embeddings

print("Token embedding first batch", token_embeddings[0])
print("\n")
print("Token embedding second batch", token_embeddings[1])
print("\n")
print("Position embeddings", position_embeddings)
print("\n")
print("Final embeddings", final_embeddings)

Tokens_ids::  tensor([[  123,   464,  4643, 11600],
        [  464,  4643, 11600,   198]])


Size of inputs:  torch.Size([2, 4])


Embedding_layer: 
 Embedding(50257, 4)


Token embedding first batch tensor([[ 1.0006, -1.4561,  2.3452,  0.9533],
        [-0.6952,  1.4324, -0.9224,  1.3483],
        [ 1.5369,  1.5365,  1.1638,  0.2416],
        [-0.2841, -1.5875,  0.7568,  1.0581]], grad_fn=<SelectBackward0>)


Token embedding second batch tensor([[-0.6952,  1.4324, -0.9224,  1.3483],
        [ 1.5369,  1.5365,  1.1638,  0.2416],
        [-0.2841, -1.5875,  0.7568,  1.0581],
        [-2.0690, -0.2814, -0.9582, -1.5726]], grad_fn=<SelectBackward0>)


Position embeddings tensor([[-1.3356,  0.4003,  1.1025,  0.3584],
        [-0.4543, -0.3627, -1.7934, -0.4577],
        [ 1.3059, -0.2482, -1.7164, -0.3580],
        [ 0.5924,  0.2455,  0.3054,  0.3135]], grad_fn=<EmbeddingBackward0>)


Final embeddings tensor([[[-0.3349, -1.0558,  3.4477,  1.3117],
         [-1.1494,  1.0697, -2.7158,  0.89

# Embedding

This code is responsible for transforming the token_ids from the tensors into embeddings.

For this example I used `batch_size = 2` to make the prints easier to understand.

Creating embeddings is basically translating a token_id into a vector.

To do this we create a matrix (almost like a matrix function) where, given an input token_id, it returns an embedding vector for that token_id.

That is what the `token_embedding_layer` is used for.

This embedding layer is created with the vocabulary size and a dimension that we choose.

In our example, the embedding layer has size:

```text
(50257 x 4)
```

The dimension `4` was chosen only to make the prints easier to visualize.

The embedding layer must have the vocabulary size because every token in the vocabulary needs to have its own embedding vector.

And `50257` is the vocabulary size provided by :contentReference[oaicite:0]{index=0}.

At this point we already have the embeddings that will later be used as the neural network input.

However, we still have a problem, or more specifically, something that will become a problem later.

What we did so far does not consider the token position inside the sequence.

So if we have two equal tokens in different positions of the same sequence, both will receive the exact same embedding.

To solve this we create a second embedding layer where the embeddings represent positions instead of tokens.

This second layer has size:

```text
(context_size x dimension)
```

`context_size` because we want one positional embedding for each position in the sequence.

And `dimension` because it must have the same embedding size as the `token_embedding_layer`.

Having the same dimension makes it possible to sum both embeddings together.

After creating these two embedding layers, we:

1. Pass the token_ids through the `token_embedding_layer`
2. Pass the token positions through the `position_embedding_layer`
3. Sum both resulting embeddings

This final result is the embedding that will be used as the neural network input.

The final embedding shape is:

```text
(batch_size, context_length, dim)
```

As we can see from the print:

```text
Token embedding first batch tensor([
 [ 1.0006, -1.4561,  2.3452,  0.9533],
 [-0.6952,  1.4324, -0.9224,  1.3483],
 [ 1.5369,  1.5365,  1.1638,  0.2416],
 [-0.2841, -1.5875,  0.7568,  1.0581]
], grad_fn=...)
```

```text
Token embedding second batch tensor([
 [-0.6952,  1.4324, -0.9224,  1.3483],
 [ 1.5369,  1.5365,  1.1638,  0.2416],
 [-0.2841, -1.5875,  0.7568,  1.0581],
 [-2.0690, -0.2814, -0.9582, -1.5726]
], grad_fn=...)
```

We have 2 tensors because `batch_size = 2`.

Inside each tensor we have 4 vectors because `context_size = 4`.

And each vector has size 4 because the embedding dimension is 4.

So if we have:

```python
batch_size = 1
```

and the DataLoader returns:

```text
tensor([[123, 464, 4643, 11600]])
```

after passing it through the embedding layer we get something like:

```text
tensor([
 [
  [ 1.0006, -1.4561,  2.3452,  0.9533],
  [-0.6952,  1.4324, -0.9224,  1.3483],
  [ 1.5369,  1.5365,  1.1638,  0.2416],
  [-0.2841, -1.5875,  0.7568,  1.0581]
 ]
], grad_fn=...)
```

Where:

```text
[ 1.0006, -1.4561, 2.3452, 0.9533 ]
```

represents the embedding vector of token_id `123`.